In [1]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

from torch.utils.tensorboard import SummaryWriter

# ================= CONFIG =================
SEED = 42

BATCH_SIZE = 256
EPOCHS = 10
LR = 3e-4
IMG_SIZE = 224

NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

2026-02-27 13:58:18.267653: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772200698.471757      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772200698.535395      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772200699.042893      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772200699.042940      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772200699.042944      55 computation_placer.cc:177] computation placer alr

CUDA available: True
GPU count: 2


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [2]:
NO_PATHS = [
    "/kaggle/input/datasets/navoneel/brain-mri-images-for-brain-tumor-detection/no",
    "/kaggle/input/datasets/ahmedhamada0/brain-tumor-detection/no"
]

YES_PATHS = [
    "/kaggle/input/datasets/navoneel/brain-mri-images-for-brain-tumor-detection/yes",
    "/kaggle/input/datasets/ahmedhamada0/brain-tumor-detection/yes"
]

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".tif")

def collect(paths, label):
    items = []
    for path in paths:
        for root, _, files in os.walk(path):
            for f in files:
                if f.lower().endswith(VALID_EXTENSIONS):
                    items.append((os.path.join(root, f), label))
    return items

data = collect(NO_PATHS, 0) + collect(YES_PATHS, 1)
df = pd.DataFrame(data, columns=["path", "label"])

print("Total Images:", len(df))
print("Class Distribution:")
print(df["label"].value_counts())

Total Images: 3253
Class Distribution:
label
1    1655
0    1598
Name: count, dtype: int64


In [3]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 2277
Val: 488
Test: 488


In [4]:
class BrainDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "path"]
        label = self.df.loc[idx, "label"]
        
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Failed loading image: {img_path}") from e
        
        if self.transform:
            img = self.transform(img)
            
        return img, torch.tensor(label, dtype=torch.float32)

In [5]:
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

train_loader = DataLoader(
    BrainDataset(train_df, train_tfms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=True
)

val_loader = DataLoader(
    BrainDataset(val_df, val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=False
)

test_loader = DataLoader(
    BrainDataset(test_df, val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=False
)

In [6]:
model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 1)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = nn.DataParallel(model)

model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 128MB/s] 


Using 2 GPUs


In [7]:
def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    loop = tqdm(
        loader,
        total=len(loader),
        desc="Train" if training else "Val/Test",
        leave=False
    )

    for images, labels in loop:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=True):
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)

        if training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item()

        probs = torch.sigmoid(outputs).detach()
        all_preds.append(probs.cpu())
        all_labels.append(labels.detach().cpu())

        loop.set_postfix(loss=f"{loss.item():.4f}")

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc = accuracy_score(all_labels, all_preds > 0.5)
    auc = roc_auc_score(all_labels, all_preds)

    return total_loss / len(loader), acc, auc

In [8]:
log_dir = f"/kaggle/working/runs/{datetime.now().strftime('%Y%m%d-%H%M%S')}"
writer = SummaryWriter(log_dir)

best_auc = 0.0

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")

    train_loss, train_acc, train_auc = run_epoch(train_loader, True)
    val_loss, val_acc, val_auc = run_epoch(val_loader, False)

    scheduler.step()

    print(f"Train | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | AUC: {train_auc:.4f}")
    print(f"Val   | Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f}")

    writer.add_scalar("Loss/Train", train_loss, epoch)
    writer.add_scalar("Loss/Val", val_loss, epoch)
    writer.add_scalar("AUC/Train", train_auc, epoch)
    writer.add_scalar("AUC/Val", val_auc, epoch)

    if val_auc > best_auc:
        best_auc = val_auc
        state_dict = model.module.state_dict() if torch.cuda.device_count() > 1 else model.state_dict()
        torch.save(state_dict, "/kaggle/working/best_mobilenetv3_large.pth")
        print("Best model saved.")

writer.close()


===== Epoch 1/10 =====


Train:   0%|          | 0/8 [00:00<?, ?it/s]

Val/Test:   0%|          | 0/2 [00:00<?, ?it/s]

Train | Loss: 0.3998 | Acc: 0.8169 | AUC: 0.9172
Val   | Loss: 0.5835 | Acc: 0.6988 | AUC: 0.9519
Best model saved.

===== Epoch 2/10 =====


Train:   0%|          | 0/8 [00:00<?, ?it/s]

...

Val/Test:   0%|          | 0/2 [00:00<?, ?it/s]

Train | Loss: 0.0010 | Acc: 1.0000 | AUC: 1.0000
Val   | Loss: 0.4451 | Acc: 0.8750 | AUC: 0.9992
Best model saved.

===== Epoch 10/10 =====


Train:   0%|          | 0/8 [00:00<?, ?it/s]

Val/Test:   0%|          | 0/2 [00:00<?, ?it/s]

Train | Loss: 0.0022 | Acc: 0.9990 | AUC: 1.0000
Val   | Loss: 0.3857 | Acc: 0.8934 | AUC: 0.9993
Best model saved.


In [9]:
state_dict = torch.load("/kaggle/working/best_mobilenetv3_large.pth")

if torch.cuda.device_count() > 1:
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()

test_loss, test_acc, test_auc = run_epoch(test_loader, training=False)

print("\n===== FINAL UNSEEN TEST RESULTS =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

Val/Test:   0%|          | 0/2 [00:00<?, ?it/s]


===== FINAL UNSEEN TEST RESULTS =====
Test Loss: 0.3595
Test Accuracy: 0.9139
Test AUC: 0.9982


In [10]:
final_state = model.module.state_dict() if torch.cuda.device_count() > 1 else model.state_dict()
torch.save(final_state, "/kaggle/working/final_mobilenetv3_large_brain_tumor.pth")

print("Final model saved.")

Final model saved.
